In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
df = pd.read_excel("data/raw/Aggregated_Flood_Data.xlsx", sheet_name="DesInventar")

df["Year"] = pd.to_datetime(df["Date"], errors="coerce").dt.year
df = df[df["Year"].between(1900, 2026)].dropna(subset=["Deaths"])

numeric_features = [
    "Year", "GEO_LAT", "GEO_LON", "Duration (d)",
    "Injured", "Missing", "Directly affected", "Indirectly Affected",
    "Relocated", "Evacuated", "Houses Destroyed", "Houses Damaged", "Losses $USD"
]
categorical_features = ["Country"]

X = df[numeric_features + categorical_features]
y = (df["Deaths"] >= 1).astype(int)

X.shape, y.value_counts(normalize=True)

((113092, 14),
 Deaths
 0    0.893494
 1    0.106506
 Name: proportion, dtype: float64)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=522
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))

(90473, 14) (22619, 14)
0.1065 0.1065


In [7]:
preprocessor = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical_features),
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

X_train_proc.shape, X_test_proc.shape

((90473, 101), (22619, 101))